# イベントストリーム生成

Lakehouseのリファレンスデータを読み込み、リアルタイムイベントストリームを生成してEventhouse（KQL Database）に取り込みます。

 **生成されるイベント:**
 1. **SensorReadingEvent** — センサーの定期計測値（温度・圧力・pH・振動・流量）
 2. **ProcessAlarmEvent** — 閾値超過によるプロセスアラーム
 3. **EquipmentStatusEvent** — 設備の状態変更（運転・待機・故障・メンテナンス）
 4. **BatchPhaseTransitionEvent** — バッチ工程のフェーズ遷移
 5. **QualityInspectionEvent** — 製造中のインライン品質検査

 **デモシナリオ:**
 - 温度逸脱: 反応器の温度がアラーム閾値を超過
 - 設備トリップ: 重要設備の緊急停止
 - 品質逸脱: 製造バッチの品質検査不合格
 - 圧力スパイク: 容器内の急激な圧力上昇
 - バッチ中止: 連鎖的な異常によるバッチ停止

 **前提条件:**
 - このノートブックを `[Prefix]_lh_chemical` Lakehouse にアタッチすること
 - 下記の KUSTO_URI と KUSTO_DATABASE を設定すること

 全セルを上から順に実行してください。

# 設定

### KUSTO_URI の取得方法
1. Fabricポータルで→ Workspace → Eventhouse → 対象の KQL Databaseを開く
2. 右側の 「Database details」 カードの Query URI をコピーします。


In [ ]:
KUSTO_URI = "https://trd-9ebjfdnrmssvdapfjj.z3.kusto.fabric.microsoft.com" #"<YOUR_EVENTHOUSE_QUERY_URI>" # 例：https://trd-9ebjfdnrmssvdapfjj.z3.kusto.fabric.microsoft.com
KUSTO_DATABASE = "MS_chemical_DB" #"<YOUR_KQL_DATABASE_NAME>" # [Prefix]_chemical_db

WORKSPACE_ID= "d995c964-321b-4486-9a11-fc23428ef52c"
LAKEHOUSE_ID ="912ed674-bdf6-4240-82ae-dd0556e63a9d"

# 依存パッケージのインストール

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install",
                       "azure-kusto-data",
                       "azure-kusto-ingest",
                       "azure-identity==1.16.0",
                       "requests>=2.32.3",
                       "--quiet"])

# シミュレーション設定
シミュレーションの実行時間、センサー計測感覚、デモシナリオの注入、Eventhouse取り込みバッチサイズ、イベントをコンソール出力するかを設定します。

In [ ]:
import json
import uuid
import random
import time
from datetime import datetime, timedelta

# シミュレーションパラメータ
SIMULATION_DURATION_MINUTES = 1      # シミュレーション実行時間（分）
SENSOR_READING_INTERVAL_SECONDS = 10 # センサー計測間隔（秒）
SCENARIO_INJECTION_ENABLED = True    # デモシナリオの注入
BATCH_SIZE = 100                     # Kusto 取り込みバッチサイズ
PRINT_EVENTS = True                  # イベントをコンソールに出力

random.seed(None)

# Eventhouse (KQL Database)への接続

In [ ]:
from azure.kusto.data import KustoClient, KustoConnectionStringBuilder, DataFormat
from azure.kusto.ingest import QueuedIngestClient, IngestionProperties
import io
import pandas as pd

KUSTO_TOKEN_SCOPE = "https://kusto.kusto.windows.net"

def get_fabric_token():
    """Fabric 組み込み認証プロバイダーからアクセストークンを取得"""
    return mssparkutils.credentials.getToken(KUSTO_TOKEN_SCOPE)

kcsb = KustoConnectionStringBuilder.with_token_provider(
    KUSTO_URI, get_fabric_token
)
kusto_client = KustoClient(kcsb)

ingest_uri = KUSTO_URI.replace("https://", "https://ingest-")
ingest_kcsb = KustoConnectionStringBuilder.with_token_provider(
    ingest_uri, get_fabric_token
)
ingest_client = QueuedIngestClient(ingest_kcsb)

result = kusto_client.execute(KUSTO_DATABASE, ".show database schema | count")
for row in result.primary_results[0]:
    print(f"✓ Eventhouse に接続しました: {KUSTO_URI}")
    print(f"  Database: {KUSTO_DATABASE}")

# 取り込みヘルパー

In [ ]:
# 各テーブルのカラム順序（.create-merge-table スキーマと一致させること）
TABLE_COLUMNS = {
    "SensorReadingEvent": [
        "event_id", "event_type", "timestamp", "source",
        "sensor_id", "equipment_id", "production_line_id", "process_order_id",
        "tag_name", "measurement_type", "value", "unit",
        "normal_min", "normal_max", "alarm_low", "alarm_high",
        "is_within_normal",
    ],
    "ProcessAlarmEvent": [
        "event_id", "event_type", "timestamp", "source",
        "sensor_id", "equipment_id", "production_line_id", "process_order_id",
        "tag_name", "alarm_type", "severity",
        "threshold_value", "actual_value", "deviation_amount",
        "action_taken",
    ],
    "EquipmentStatusEvent": [
        "event_id", "event_type", "timestamp", "source",
        "equipment_id", "production_line_id",
        "equipment_name", "equipment_type",
        "previous_status", "new_status", "reason",
    ],
    "BatchPhaseTransitionEvent": [
        "event_id", "event_type", "timestamp", "source",
        "process_order_id", "batch_number", "product_id", "production_line_id",
        "previous_phase", "new_phase", "sequence_number",
        "set_temperature", "set_pressure",
        "actual_temperature", "actual_pressure",
    ],
    "QualityInspectionEvent": [
        "event_id", "event_type", "timestamp", "source",
        "process_order_id", "batch_number", "product_id",
        "inspection_item", "measured_value",
        "spec_lower", "spec_upper", "pass_fail",
        "lot_number",
    ],
}

_event_buffers = {table: [] for table in TABLE_COLUMNS}
_ingest_counts = {table: 0 for table in TABLE_COLUMNS}

def flatten_event(event):
    row = {
        "event_id": event["event_id"],
        "event_type": event["event_type"],
        "timestamp": event["timestamp"],
        "source": event["source"],
    }
    row.update(event["body"])
    return row

def buffer_event(event):
    table = event["event_type"]
    row = flatten_event(event)
    _event_buffers[table].append(row)
    if len(_event_buffers[table]) >= BATCH_SIZE:
        flush_buffer(table)

def flush_buffer(table):
    rows = _event_buffers[table]
    if not rows:
        return
    columns = TABLE_COLUMNS[table]
    df = pd.DataFrame(rows, columns=columns)
    props = IngestionProperties(
        database=KUSTO_DATABASE,
        table=table,
        data_format=DataFormat.CSV,
    )
    csv_buffer = io.StringIO()
    df.to_csv(csv_buffer, index=False, header=False)
    csv_buffer.seek(0)
    csv_bytes = io.BytesIO(csv_buffer.getvalue().encode("utf-8"))
    ingest_client.ingest_from_stream(csv_bytes, ingestion_properties=props)
    _ingest_counts[table] += len(rows)
    _event_buffers[table] = []

def flush_all_buffers():
    for table in TABLE_COLUMNS:
        flush_buffer(table)

# Lakehouse からリファレンスデータを読み込み

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
lakehouse_path = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{LAKEHOUSE_ID}/Tables/dbo"
#lakehouse_path = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{LAKEHOUSE_ID}/Tables" #スキーマが無い場合

production_lines_df = spark.read.format("delta").load(f"{lakehouse_path}/production_lines").toPandas()
equipment_df        = spark.read.format("delta").load(f"{lakehouse_path}/equipment").toPandas()
sensors_df          = spark.read.format("delta").load(f"{lakehouse_path}/sensors").toPandas()
products_df         = spark.read.format("delta").load(f"{lakehouse_path}/products").toPandas()
process_orders_df   = spark.read.format("delta").load(f"{lakehouse_path}/process_orders").toPandas()
operation_phases_df = spark.read.format("delta").load(f"{lakehouse_path}/operation_phases").toPandas()

# ルックアップ辞書の構築
production_lines = {row["production_line_id"]: row.to_dict() for _, row in production_lines_df.iterrows()}
equipment = {row["equipment_id"]: row.to_dict() for _, row in equipment_df.iterrows()}
sensors = {row["sensor_id"]: row.to_dict() for _, row in sensors_df.iterrows()}
products = {row["product_id"]: row.to_dict() for _, row in products_df.iterrows()}
process_orders = {row["process_order_id"]: row.to_dict() for _, row in process_orders_df.iterrows()}

# センサー → 設備 → 製造ラインの関連付け
sensor_to_equipment = {s["sensor_id"]: s["equipment_id"] for s in sensors.values()}
equipment_to_line = {e["equipment_id"]: e["production_line_id"] for e in equipment.values()}

# 実行中のプロセスオーダーを取得
active_orders = process_orders_df[process_orders_df["status"] == "running"].to_dict("records")

# 実行中オーダーに紐づくフェーズを取得
active_order_ids = {o["process_order_id"] for o in active_orders}
active_phases = operation_phases_df[
    operation_phases_df["process_order_id"].isin(active_order_ids)
].to_dict("records")

# 稼働中の設備のセンサーリスト
running_equipment_ids = {e["equipment_id"] for e in equipment.values() if e["status"] == "running"}
active_sensors = [s for s in sensors.values() if s["equipment_id"] in running_equipment_ids]

print(f"リファレンスデータを読み込みました:")
print(f"  製造ライン: {len(production_lines)}")
print(f"  設備: {len(equipment)}（稼働中: {len(running_equipment_ids)}）")
print(f"  センサー: {len(sensors)}（アクティブ: {len(active_sensors)}）")
print(f"  プロセスオーダー: {len(process_orders)}（実行中: {len(active_orders)}）")
print(f"  オペレーションフェーズ: {len(active_phases)}")

# ヘルパー関数

In [ ]:
def new_event_id():
    return str(uuid.uuid4())

def make_envelope(event_type, source, body):
    """イベント本体を標準エンベロープでラップ"""
    return {
        "event_id": new_event_id(),
        "event_type": event_type,
        "timestamp": datetime.utcnow().isoformat() + "Z",
        "source": source,
        "body": body,
    }

# アラーム種別リファレンス

In [ ]:
ALARM_TYPES = [
    {"alarm_type": "temperature_high",  "description": "温度上限超過",   "severity": "critical"},
    {"alarm_type": "temperature_low",   "description": "温度下限逸脱",   "severity": "warning"},
    {"alarm_type": "pressure_high",     "description": "圧力上限超過",   "severity": "critical"},
    {"alarm_type": "pressure_low",      "description": "圧力下限逸脱",   "severity": "warning"},
    {"alarm_type": "vibration_high",    "description": "振動値異常",     "severity": "warning"},
    {"alarm_type": "pH_out_of_range",   "description": "pH 規格外",      "severity": "critical"},
    {"alarm_type": "flow_anomaly",      "description": "流量異常",       "severity": "warning"},
    {"alarm_type": "level_alarm",       "description": "液位アラーム",   "severity": "info"},
]

EQUIPMENT_FAULT_REASONS = [
    "ベアリング過熱",
    "モーター過電流",
    "シール漏洩検知",
    "振動異常値超過",
    "インバーター故障",
    "冷却水流量低下",
    "計装エア圧力低下",
]

# プラント状態トラッカー

In [ ]:
class PlantSimState:
    """化学プラントのリアルタイムシミュレーション状態を管理"""

    def __init__(self, sensor, equip, line, order=None, phase=None):
        self.sensor = sensor
        self.equipment = equip
        self.production_line = line
        self.order = order
        self.phase = phase

        # センサー現在値（正規範囲内でランダム初期化）
        nmin = float(sensor.get("normal_min", 0))
        nmax = float(sensor.get("normal_max", 100))
        self.current_value = random.uniform(nmin, nmax)

        # シナリオ注入フラグ
        self.temp_excursion_injected = False
        self.equipment_trip_injected = False
        self.pressure_spike_injected = False

    def advance(self):
        """センサー値を小幅に変動させる"""
        nmin = float(self.sensor.get("normal_min", 0))
        nmax = float(self.sensor.get("normal_max", 100))
        spread = (nmax - nmin) * 0.05
        self.current_value += random.uniform(-spread, spread)
        # 正規範囲に緩やかに引き戻す
        midpoint = (nmin + nmax) / 2
        self.current_value += (midpoint - self.current_value) * 0.02

    @property
    def is_within_normal(self):
        nmin = float(self.sensor.get("normal_min", 0))
        nmax = float(self.sensor.get("normal_max", 100))
        return nmin <= self.current_value <= nmax

    @property
    def is_alarm(self):
        alow = float(self.sensor.get("alarm_low", -999999))
        ahigh = float(self.sensor.get("alarm_high", 999999))
        return self.current_value < alow or self.current_value > ahigh

# イベントジェネレーター

In [ ]:
def generate_sensor_reading(state):
    """SensorReadingEvent を生成"""
    s = state.sensor
    return make_envelope("SensorReadingEvent", "dcs-historian", {
        "sensor_id": s["sensor_id"],
        "equipment_id": s["equipment_id"],
        "production_line_id": state.production_line["production_line_id"],
        "process_order_id": state.order["process_order_id"] if state.order else None,
        "tag_name": s["tag_name"],
        "measurement_type": s["measurement_type"],
        "value": round(state.current_value, 3),
        "unit": s["unit"],
        "normal_min": float(s.get("normal_min", 0)),
        "normal_max": float(s.get("normal_max", 100)),
        "alarm_low": float(s.get("alarm_low", -999)),
        "alarm_high": float(s.get("alarm_high", 999)),
        "is_within_normal": state.is_within_normal,
    })


def generate_process_alarm(state, alarm_type_info):
    """ProcessAlarmEvent を生成"""
    s = state.sensor
    threshold = float(s.get("alarm_high", 999)) if "high" in alarm_type_info["alarm_type"] else float(s.get("alarm_low", -999))
    return make_envelope("ProcessAlarmEvent", "alarm-management", {
        "sensor_id": s["sensor_id"],
        "equipment_id": s["equipment_id"],
        "production_line_id": state.production_line["production_line_id"],
        "process_order_id": state.order["process_order_id"] if state.order else None,
        "tag_name": s["tag_name"],
        "alarm_type": alarm_type_info["alarm_type"],
        "severity": alarm_type_info["severity"],
        "threshold_value": threshold,
        "actual_value": round(state.current_value, 3),
        "deviation_amount": round(abs(state.current_value - threshold), 3),
        "action_taken": random.choice(["operator_notified", "auto_adjusted", "escalated"]),
    })


def generate_equipment_status(equip, line, prev_status, new_status, reason):
    """EquipmentStatusEvent を生成"""
    return make_envelope("EquipmentStatusEvent", "plc-gateway", {
        "equipment_id": equip["equipment_id"],
        "production_line_id": line["production_line_id"],
        "equipment_name": equip["name"],
        "equipment_type": equip["equipment_type"],
        "previous_status": prev_status,
        "new_status": new_status,
        "reason": reason,
    })


def generate_batch_phase_transition(order, prev_phase, new_phase, seq, line_id):
    """BatchPhaseTransitionEvent を生成"""
    return make_envelope("BatchPhaseTransitionEvent", "mes-system", {
        "process_order_id": order["process_order_id"],
        "batch_number": order["batch_number"],
        "product_id": order["product_id"],
        "production_line_id": line_id,
        "previous_phase": prev_phase,
        "new_phase": new_phase,
        "sequence_number": seq,
        "set_temperature": round(random.uniform(60, 180), 1),
        "set_pressure": round(random.uniform(0.05, 3.0), 3),
        "actual_temperature": round(random.uniform(58, 182), 1),
        "actual_pressure": round(random.uniform(0.04, 3.1), 3),
    })


def generate_quality_inspection(order, product):
    """QualityInspectionEvent を生成"""
    spec_lower = float(product.get("spec_lower", 0))
    spec_upper = float(product.get("spec_upper", 100))
    # 90% の確率で合格範囲内
    if random.random() < 0.9:
        measured = random.uniform(spec_lower, spec_upper)
        pass_fail = "pass"
    else:
        offset = (spec_upper - spec_lower) * random.uniform(0.01, 0.1)
        measured = spec_upper + offset if random.random() < 0.5 else spec_lower - offset
        pass_fail = "fail"
    inspection_items = ["purity", "viscosity", "moisture_content", "particle_size", "density", "pH"]
    return make_envelope("QualityInspectionEvent", "lims-system", {
        "process_order_id": order["process_order_id"],
        "batch_number": order["batch_number"],
        "product_id": order["product_id"],
        "inspection_item": random.choice(inspection_items),
        "measured_value": round(measured, 4),
        "spec_lower": spec_lower,
        "spec_upper": spec_upper,
        "pass_fail": pass_fail,
        "lot_number": f"L-{datetime.utcnow().strftime('%Y')}-{random.randint(1000,9999):04d}",
    })

# デモシナリオ注入

In [ ]:
def check_and_inject_scenarios(states, events):
    """
    デモシナリオの条件を確認し、必要に応じてイベントを注入する。
    """
    for state in states:
        s = state.sensor
        mtype = s.get("measurement_type", "")

        # --- シナリオ 1: 温度逸脱 ---
        if (not state.temp_excursion_injected
                and mtype == "temperature"
                and random.random() < 0.08):
            state.temp_excursion_injected = True
            alarm_high = float(s.get("alarm_high", 999))
            state.current_value = alarm_high + random.uniform(5, 20)
            alarm = next((a for a in ALARM_TYPES if a["alarm_type"] == "temperature_high"), ALARM_TYPES[0])
            print(f"\n  🚨 シナリオ: 温度逸脱 — {s['tag_name']} ({state.equipment['name']})")
            print(f"     実測値: {state.current_value:.1f}{s['unit']}, アラーム閾値: {alarm_high}{s['unit']}")
            events.append(generate_process_alarm(state, alarm))

        # --- シナリオ 2: 設備トリップ ---
        if (not state.equipment_trip_injected
                and state.equipment.get("equipment_type") in ("reactor", "compressor", "pump")
                and random.random() < 0.05):
            state.equipment_trip_injected = True
            reason = random.choice(EQUIPMENT_FAULT_REASONS)
            line = state.production_line
            print(f"\n  🚨 シナリオ: 設備トリップ — {state.equipment['name']} ({state.equipment['equipment_number']})")
            print(f"     原因: {reason}")
            print(f"     製造ライン: {line['name']}")
            events.append(generate_equipment_status(
                state.equipment, line, "running", "fault", reason
            ))

        # --- シナリオ 3: 圧力スパイク ---
        if (not state.pressure_spike_injected
                and mtype == "pressure"
                and random.random() < 0.06):
            state.pressure_spike_injected = True
            alarm_high = float(s.get("alarm_high", 999))
            state.current_value = alarm_high + random.uniform(0.2, 1.5)
            alarm = next((a for a in ALARM_TYPES if a["alarm_type"] == "pressure_high"), ALARM_TYPES[2])
            print(f"\n  🚨 シナリオ: 圧力スパイク — {s['tag_name']} ({state.equipment['name']})")
            print(f"     実測値: {state.current_value:.3f}{s['unit']}, アラーム閾値: {alarm_high}{s['unit']}")
            events.append(generate_process_alarm(state, alarm))

    # --- シナリオ 4: 品質逸脱（アクティブオーダーから）---
    if active_orders and random.random() < 0.10:
        order = random.choice(active_orders)
        product = products.get(order["product_id"], {})
        if product:
            spec_upper = float(product.get("spec_upper", 100))
            spec_lower = float(product.get("spec_lower", 0))
            offset = (spec_upper - spec_lower) * random.uniform(0.05, 0.15)
            measured = spec_upper + offset
            print(f"\n  🚨 シナリオ: 品質逸脱 — バッチ {order['batch_number']}")
            print(f"     製品: {product.get('name', 'Unknown')}, 測定値: {measured:.4f}, 規格上限: {spec_upper}")
            events.append(make_envelope("QualityInspectionEvent", "lims-system", {
                "process_order_id": order["process_order_id"],
                "batch_number": order["batch_number"],
                "product_id": order["product_id"],
                "inspection_item": "purity",
                "measured_value": round(measured, 4),
                "spec_lower": spec_lower,
                "spec_upper": spec_upper,
                "pass_fail": "fail",
                "lot_number": f"L-{datetime.utcnow().strftime('%Y')}-{random.randint(1000,9999):04d}",
            }))

    # --- シナリオ 5: バッチフェーズ遷移 ---
    if active_orders and random.random() < 0.15:
        order = random.choice(active_orders)
        phases = ["reaction", "distillation", "purification", "drying", "cooling", "mixing", "polymerization"]
        prev_phase = random.choice(phases)
        new_phase = random.choice([p for p in phases if p != prev_phase])
        print(f"\n  📋 フェーズ遷移 — バッチ {order['batch_number']}: {prev_phase} → {new_phase}")
        events.append(generate_batch_phase_transition(
            order, prev_phase, new_phase,
            random.randint(1, 5), order["production_line_id"]
        ))

# シミュレーション状態の初期化

In [ ]:
sim_states = []

for sensor in active_sensors:
    equip = equipment.get(sensor["equipment_id"])
    if not equip:
        continue
    line = production_lines.get(equip.get("production_line_id"))
    if not line:
        continue

    # このセンサーに関連するアクティブオーダーを探す
    order = None
    phase = None
    for o in active_orders:
        if o["production_line_id"] == line["production_line_id"]:
            order = o
            break

    state = PlantSimState(sensor, equip, line, order, phase)
    sim_states.append(state)

print(f"\nシミュレーション状態を初期化しました: {len(sim_states)} センサー")
for s in sim_states[:5]:
    print(f"  {s.sensor['tag_name']} ({s.sensor['measurement_type']}) "
          f"→ {s.equipment['name']} → {s.production_line['name']}")
if len(sim_states) > 5:
    print(f"  ... 他 {len(sim_states) - 5} センサー")

# イベント生成ループの実行

In [ ]:
SEPARATOR = "=" * 60
scenario_status = "ON" if SCENARIO_INJECTION_ENABLED else "OFF"
print(f"\n{SEPARATOR}")
print(f"イベント生成を開始します（{SIMULATION_DURATION_MINUTES}分間）")
print(f"アクティブセンサー: {len(sim_states)}")
print(f"計測間隔: {SENSOR_READING_INTERVAL_SECONDS}秒")
print(f"シナリオ注入: {scenario_status}")
print(f"送信先: Eventhouse ({KUSTO_DATABASE})")
print(f"{SEPARATOR}\n")

total_events = 0
start_time = time.time()
end_time = start_time + (SIMULATION_DURATION_MINUTES * 60)
tick = 0

try:
    while time.time() < end_time:
        tick += 1
        tick_events = []

        for state in sim_states:
            state.advance()

            # 1. センサー読み取りイベント（毎 tick）
            tick_events.append(generate_sensor_reading(state))

            # 2. アラーム閾値を超えた場合のアラームイベント
            if state.is_alarm:
                mtype = state.sensor.get("measurement_type", "")
                matching_alarms = [a for a in ALARM_TYPES if mtype in a["alarm_type"]]
                if matching_alarms:
                    tick_events.append(generate_process_alarm(state, random.choice(matching_alarms)))

            # 3. ランダムな品質検査イベント
            if state.order and random.random() < 0.03:
                product = products.get(state.order["product_id"])
                if product:
                    tick_events.append(generate_quality_inspection(state.order, product))

        # 4. デモシナリオの注入
        if SCENARIO_INJECTION_ENABLED:
            check_and_inject_scenarios(sim_states, tick_events)

        # バッファにイベントを追加
        for event in tick_events:
            if PRINT_EVENTS:
                body = event["body"]
                tag = body.get("tag_name", body.get("equipment_name", body.get("batch_number", "N/A")))
                print(f"  [{event['event_type']}] {tag}")
            buffer_event(event)

        total_events += len(tick_events)

        if tick % 6 == 0:
            flush_all_buffers()
            elapsed = time.time() - start_time
            print(f"\n--- Tick {tick} | {elapsed:.0f}秒経過 | {total_events} イベント生成済 ---")
            print(f"    取り込み数: {dict(_ingest_counts)}")

        time.sleep(SENSOR_READING_INTERVAL_SECONDS)

except KeyboardInterrupt:
    print("\n\n⚠ シミュレーションがユーザーにより停止されました。")

finally:
    flush_all_buffers()
    elapsed = time.time() - start_time
    print(f"\n{SEPARATOR}")
    print(f"シミュレーション完了")
    print(f"  実行時間: {elapsed:.0f}秒 ({elapsed/60:.1f}分)")
    print(f"  生成イベント合計: {total_events}")
    print(f"  テーブル別取り込み数:")
    for table, count in _ingest_counts.items():
        print(f"    {table:<35} {count:>6}")
    print(f"{SEPARATOR}")


# 検証：Eventhouse（KQL Database)テーブルのクエリ
上のブロックが実行されてから、各KQL Databaseのテーブルに値が反映されるまで少しタイムラグがあります。
これは、APIからの書き込みとして処理がされているからで、実際のストリーミングデータの場合には発生しません。

In [ ]:
print("Eventhouse 行数（Queued Ingestion の完了まで数分かかる場合があります）:\n")
for table in TABLE_COLUMNS:
    try:
        result = kusto_client.execute(KUSTO_DATABASE, f"{table} | count")
        for row in result.primary_results[0]:
            print(f"  {table:<35} {row[0]:>6} rows")
    except Exception as e:
        print(f"  {table:<35} ERROR: {e}")

In [ ]:
# 最新のセンサーリーディング（センサーごと）
print("\n最新のセンサーリーディング:")
query = """
SensorReadingEvent
| summarize arg_max(timestamp, *) by sensor_id
| project timestamp, tag_name, measurement_type, value, unit, is_within_normal
| order by timestamp desc
| take 10
"""
result = kusto_client.execute(KUSTO_DATABASE, query)
for row in result.primary_results[0]:
    status = "✓" if row["is_within_normal"] else "⚠"
    print(f"  {status} {row['tag_name']} ({row['measurement_type']}): "
          f"{row['value']} {row['unit']}")

# イベントサマリー
イベントは Kusto Python SDK 経由で Eventhouse テーブルに直接取り込まれます。
Queued Ingestion の場合、完全に反映されるまで 2〜5 分かかることがあります。

### シナリオ別 KQL クエリ:
- **温度逸脱**: `ProcessAlarmEvent | where alarm_type == "temperature_high"`
- **設備トリップ**: `EquipmentStatusEvent | where new_status == "fault"`
- **品質逸脱**: `QualityInspectionEvent | where pass_fail == "fail"`
- **圧力スパイク**: `ProcessAlarmEvent | where alarm_type == "pressure_high"`
- **フェーズ遷移**: `BatchPhaseTransitionEvent | summarize count() by new_phase`